# Notebook 3: SQL Analysis
>
> We load the cleaned CSV into an in-memory SQLite database, then query it with `pd.read_sql()`.

In [1]:
import pandas as pd
import sqlite3
import os

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

rides   = pd.read_csv('data/processed/rides_clean.csv', parse_dates=['ride_date'])
drivers = pd.read_csv('data/raw/drivers.csv',           parse_dates=['join_date'])
zones   = pd.read_csv('data/raw/zones.csv')

conn = sqlite3.connect(':memory:')
rides.to_sql('rides',   conn, index=False, if_exists='replace')
drivers.to_sql('drivers', conn, index=False, if_exists='replace')
zones.to_sql('zones',   conn, index=False, if_exists='replace')

print('Tables loaded into SQLite:')
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)

Tables loaded into SQLite:


,name
0,rides
1,drivers
2,zones


## Query 1 — Revenue by Zone
**Business question**: Which pickup zones generate the most value?

In [2]:
q1 = """
SELECT
    pickup_zone,
    COUNT(ride_id)             AS total_rides,
    ROUND(SUM(fare_amount), 0) AS total_revenue,
    ROUND(AVG(fare_amount), 2) AS avg_fare,
    ROUND(AVG(rating), 2)     AS avg_rating
FROM rides
WHERE ride_status = 'Completed'
GROUP BY pickup_zone
ORDER BY total_revenue DESC
"""
pd.read_sql(q1, conn)

,pickup_zone,total_rides,total_revenue,avg_fare,avg_rating
0,Yeshwanthpur,3133,242032.0,77.25,4.33
1,Marathahalli,3166,238593.0,75.36,4.33
2,HSR Layout,3033,237828.0,78.41,4.34
3,Jayanagar,3077,234547.0,76.23,4.32
4,Electronic City,3026,233922.0,77.30,4.32
5,Koramangala,3106,233751.0,75.26,4.32
6,Indiranagar,3052,233015.0,76.35,4.31
7,BTM Layout,3036,230989.0,76.08,4.32
8,MG Road,3043,227145.0,74.65,4.33
9,Whitefield,2967,226633.0,76.38,4.32


## Query 2 — Cancellation Rate by Hour
**Business question**: When are we failing users most? (Supply-demand gap)

In [3]:
q2 = """
SELECT
    ride_hour,
    COUNT(*) AS total_rides,
    SUM(CASE WHEN ride_status = 'Cancelled' THEN 1 ELSE 0 END) AS cancelled,
    ROUND(
        100.0
        * SUM(CASE WHEN ride_status = 'Cancelled' THEN 1 ELSE 0 END)
        / COUNT(*)
    , 2) AS cancellation_rate_pct
FROM rides
GROUP BY ride_hour
ORDER BY cancellation_rate_pct DESC
"""
pd.read_sql(q2, conn).head(10)

,ride_hour,total_rides,cancelled,cancellation_rate_pct
0,2,239,41,17.15
1,14,1389,213,15.33
2,17,3349,508,15.17
3,19,4312,634,14.70
4,1,472,69,14.62
5,3,226,33,14.60
6,13,1416,206,14.55
7,5,488,71,14.55
8,16,1936,281,14.51
9,18,4332,625,14.43


## Query 3 — Weekend vs Weekday
**Business question**: Should we run different shift policies on weekends?

In [4]:
q3 = """
SELECT
    CASE WHEN is_weekend = 1 THEN 'Weekend' ELSE 'Weekday' END AS day_type,
    COUNT(*)                    AS total_rides,
    ROUND(AVG(fare_amount), 2) AS avg_fare,
    ROUND(AVG(distance_km), 2) AS avg_distance_km,
    ROUND(AVG(rating), 2)     AS avg_rating
FROM rides
WHERE ride_status = 'Completed'
GROUP BY is_weekend
"""
pd.read_sql(q3, conn)

,day_type,total_rides,avg_fare,avg_distance_km,avg_rating
0,Weekday,26096,76.27,5.08,4.33
1,Weekend,10519,76.16,5.06,4.32


## Query 4 — Driver Performance (JOIN)
**Business question**: Who are our top earners and who cancels the most?

In [5]:
q4 = """
SELECT
    r.driver_id,
    d.vehicle_type,
    COUNT(r.ride_id)                                              AS total_rides,
    ROUND(SUM(r.fare_amount), 0)                                 AS total_earnings,
    ROUND(AVG(r.rating), 2)                                     AS avg_rating,
    SUM(CASE WHEN r.ride_status = 'Cancelled' THEN 1 ELSE 0 END) AS cancellations
FROM rides r
JOIN drivers d ON r.driver_id = d.driver_id
WHERE r.ride_status IN ('Completed', 'Cancelled')
GROUP BY r.driver_id, d.vehicle_type
HAVING total_rides > 10
ORDER BY total_earnings DESC
LIMIT 15
"""
pd.read_sql(q4, conn)

,driver_id,vehicle_type,total_rides,total_earnings,avg_rating,cancellations
0,4087,Bike,20,2456.0,3.58,6
1,4840,Bike,16,1966.0,3.81,3
2,1466,Bike,15,1838.0,4.07,1
3,1910,Bike,17,1824.0,4.00,3
4,1178,Bike,14,1809.0,3.43,4
5,3033,Bike,21,1757.0,3.64,5
6,1289,Bike,17,1699.0,3.18,7
7,3205,Auto,16,1697.0,4.47,1
8,4513,Bike,18,1640.0,4.14,2
9,4280,Bike,16,1638.0,4.25,1


## Query 5 — Popular Routes
**Business question**: Where should we pre-position drivers?

In [6]:
q5 = """
SELECT
    pickup_zone,
    dropoff_zone,
    COUNT(*)                   AS ride_count,
    ROUND(AVG(distance_km), 2) AS avg_distance_km,
    ROUND(AVG(fare_amount), 2) AS avg_fare
FROM rides
WHERE ride_status = 'Completed'
GROUP BY pickup_zone, dropoff_zone
ORDER BY ride_count DESC
LIMIT 15
"""
pd.read_sql(q5, conn)

,pickup_zone,dropoff_zone,ride_count,avg_distance_km,avg_fare
0,Marathahalli,Electronic City,321,4.88,73.85
1,Yeshwanthpur,HSR Layout,320,5.36,78.47
2,Koramangala,Indiranagar,317,5.01,76.26
3,Indiranagar,Whitefield,309,5.34,79.22
4,Marathahalli,BTM Layout,307,5.34,79.93
5,MG Road,Whitefield,306,4.93,73.82
6,Indiranagar,Koramangala,301,5.48,81.49
7,Marathahalli,Bellandur,301,5.13,77.18
8,HSR Layout,Marathahalli,300,5.29,78.90
9,Jayanagar,Indiranagar,299,5.45,80.18


## Query 6 — Monthly Revenue Trend
**Business question**: Are we growing month-over-month?

In [7]:
q6 = """
SELECT
    STRFTIME('%Y-%m', ride_date) AS month,
    COUNT(*)                     AS total_rides,
    ROUND(SUM(fare_amount), 0)  AS monthly_revenue,
    ROUND(AVG(fare_amount), 2)  AS avg_fare
FROM rides
WHERE ride_status = 'Completed'
GROUP BY month
ORDER BY month
"""
pd.read_sql(q6, conn)

,month,total_rides,monthly_revenue,avg_fare
0,2024-01,3095,242877.0,78.47
1,2024-02,2896,221423.0,76.46
2,2024-03,3177,239834.0,75.49
3,2024-04,2932,217384.0,74.14
4,2024-05,3066,230100.0,75.05
5,2024-06,2986,226237.0,75.77
6,2024-07,3206,247169.0,77.10
7,2024-08,3086,236280.0,76.57
8,2024-09,2888,224319.0,77.67
9,2024-10,3103,235159.0,75.78


## Query 7 — Payment Mode Breakdown
**Business question**: Is our UPI concentration a risk?

In [8]:
q7 = """
SELECT
    payment_mode,
    COUNT(*) AS total_rides,
    ROUND(
        100.0 * COUNT(*) /
        (SELECT COUNT(*) FROM rides WHERE ride_status = 'Completed')
    , 2) AS pct_of_total,
    ROUND(AVG(fare_amount), 2) AS avg_fare
FROM rides
WHERE ride_status = 'Completed'
GROUP BY payment_mode
ORDER BY total_rides DESC
"""
pd.read_sql(q7, conn)

,payment_mode,total_rides,pct_of_total,avg_fare
0,UPI,19989,54.59,75.63
1,Cash,11028,30.12,76.92
2,Card,5598,15.29,77.06


## Bonus — Business Insights Summary

In [9]:
insights = [
    ('Peak demand window',   'Evening Rush (6-9 PM) has 2x more rides than Late Night'),
    ('Top revenue zone',     'Koramangala & Whitefield lead in completed ride revenue'),
    ('Cancellation risk',    'Highest cancellation rate seen at 8-9 AM morning rush'),
    ('Weekend pricing',      'Weekend avg fare ~INR 12 higher than weekday'),
    ('Payment risk',         'UPI = 55% of payments — single point of failure'),
    ('Residential gap',      'No-driver rides spike post 9 PM in residential zones'),
]

df_insights = pd.DataFrame(insights, columns=['Insight', 'Finding'])
df_insights.index = df_insights.index + 1
print(df_insights.to_string())

conn.close()
print('\nSQL connection closed.')

              Insight                                                  Finding
1  Peak demand window  Evening Rush (6-9 PM) has 2x more rides than Late Night
2    Top revenue zone  Koramangala & Whitefield lead in completed ride revenue
3   Cancellation risk    Highest cancellation rate seen at 8-9 AM morning rush
4     Weekend pricing             Weekend avg fare ~INR 12 higher than weekday
5        Payment risk          UPI = 55% of payments — single point of failure
6     Residential gap     No-driver rides spike post 9 PM in residential zones

SQL connection closed.
